# Laboratorio 11.6 · Preparando un dataset real
**Redes Neuronales en Mortales**, Capítulo II, sección 11.6

En este cuaderno vas a preparar, paso a paso, un conjunto de datos **real** para que una red neuronal pueda aprender de él: las medidas de 344 pingüinos de tres especies, registradas por científicos en la Antártida.

La pregunta es: **¿se puede saber la especie de un pingüino solo con sus medidas?**

Antes de responderla, hay que hacer el trabajo que casi nadie ve: mirar, limpiar, separar, transformar y comprobar.

**Cómo usarlo:** ejecuta cada celda en orden con el botón ▶ (o Mayúsculas + Enter). No necesitas instalar nada.

In [ ]:
# Herramientas que vamos a usar (ya vienen instaladas en Google Colab)
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

# Los datos de los pingüinos de Palmer (Horst, Hill y Gorman, 2020)
df = sns.load_dataset("penguins")
print("Listo: datos cargados")

## Paso 1 · Mirar antes de tocar
Nunca se empieza a limpiar sin saber qué hay. ¿Cuántas filas y columnas? ¿Qué significa cada columna?

| Columna | Qué es | Tipo |
|---|---|---|
| species | Especie: Adelie, Chinstrap o Gentoo | **Etiqueta** (lo que queremos predecir) |
| island | Isla donde se registró | Categórico nominal |
| bill_length_mm, bill_depth_mm | Largo y alto del pico (mm) | Numérico continuo |
| flipper_length_mm | Largo de la aleta (mm) | Numérico continuo |
| body_mass_g | Peso (gramos) | Numérico continuo |
| sex | Sexo | Categórico nominal |

In [ ]:
print("Filas y columnas:", df.shape)
df.head()

In [ ]:
# Resumen de las columnas numéricas: fíjate en los rangos
df.describe().round(1)

👀 Mira las filas **min** y **max**: el pico mide entre 13 y 60 mm, pero el peso va de 2 700 a 6 300 gramos. Si se lo entregamos así a una red, el peso dominará todas las cuentas. Lo resolveremos en el paso 6.

## Paso 2 · Valores faltantes
Una red neuronal no acepta celdas vacías: un solo valor vacío (`NaN`) contamina todas las cuentas en las que participa. Hay que encontrarlos y decidir qué hacer.

In [ ]:
df.isna().sum()

In [ ]:
# Dos pingüinos no tienen ninguna medida: no hay nada que aprender de ellos, se eliminan
sin_medidas = df["bill_length_mm"].isna()
print("Filas sin medidas:", sin_medidas.sum())
df = df[~sin_medidas].copy()

# Al resto le falta solo el sexo. No inventamos: lo marcamos como "Desconocido"
print("Pingüinos sin sexo registrado:", df["sex"].isna().sum())
df["sex"] = df["sex"].fillna("Desconocido")

print("Filas que quedan:", len(df))
print("Celdas vacías que quedan:", df.isna().sum().sum())

## Paso 3 · Duplicados

In [ ]:
print("Filas repetidas:", df.duplicated().sum())

Este conjunto no tiene filas repetidas. En fotos o textos es muy común tener casi-duplicados (la misma hoja fotografiada dos veces), y son peligrosos si uno cae en entrenamiento y otro en prueba.

## Paso 4 · Separar la etiqueta
La **etiqueta** es la respuesta correcta que la red debe aprender. Aquí es la especie. Todo lo demás son las **características** de entrada.

In [ ]:
y = df["species"]
X = df.drop(columns="species")
print("¿Cuántos pingüinos hay de cada especie?")
print(y.value_counts())

Las clases no están equilibradas: hay más del doble de Adelie que de Chinstrap. Por eso, al dividir, conviene **estratificar**.

## Paso 5 · Dividir PRIMERO
Antes de escalar o transformar nada, separamos los datos en tres grupos: **entrenamiento (70 %)**, **validación (15 %)** y **prueba (15 %)**. Con `stratify` cada grupo conserva la misma proporción de especies.

In [ ]:
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.15/0.85, stratify=y_tmp, random_state=42)

print("Entrenamiento:", len(X_train), "| Validación:", len(X_val), "| Prueba:", len(X_test))
pd.DataFrame({
    "entrenamiento %": (y_train.value_counts(normalize=True)*100).round(1),
    "validación %": (y_val.value_counts(normalize=True)*100).round(1),
    "prueba %": (y_test.value_counts(normalize=True)*100).round(1),
})

## Paso 6 · Transformar con las reglas del entrenamiento
- Las categorías (isla y sexo) pasan a **one-hot** (sección 5.3).
- Las medidas se **estandarizan**: a cada valor se le resta el promedio y se divide entre la desviación estándar.

⚠️ El promedio y la desviación se calculan **solo con el entrenamiento** (`fit`). Después se usan esos mismos números para transformar validación y prueba (`transform`). Si los calcularas con todos los datos, la prueba se filtraría al entrenamiento: eso es **fuga de datos**.

In [ ]:
num = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
cat = ["island", "sex"]

escala = StandardScaler().fit(X_train[num])     # aprende SOLO del entrenamiento
print("Promedios del entrenamiento:", {c: float(v) for c, v in zip(num, escala.mean_.round(1))})
print("Desviaciones del entrenamiento:", {c: float(v) for c, v in zip(num, escala.scale_.round(1))})

def preparar(X, escalar=True):
    medidas = X[num]
    if escalar:
        medidas = pd.DataFrame(escala.transform(medidas), columns=num, index=X.index)
    categorias = pd.get_dummies(X[cat]).astype(int)
    return pd.concat([medidas, categorias], axis=1)

columnas = preparar(X_train).columns
listo = lambda X, escalar=True: preparar(X, escalar).reindex(columns=columnas, fill_value=0)

Xtr, Xva, Xte = listo(X_train), listo(X_val), listo(X_test)
print("\nCada pingüino ahora es un vector de", Xtr.shape[1], "números:")
Xtr.head(3).round(2)

Fíjate: 6 columnas de características se convirtieron en **10 números** por pingüino (4 medidas + 3 islas + 3 valores de sexo). Esos 10 números son la capa de entrada de la red (sección 12.1).

## Paso 7 · ¿Valió la pena?
Entrenamos la misma red pequeña (una capa oculta de 8 neuronas) dos veces: con las medidas **escaladas** y **sin escalar**. Para que sea justo, repetimos cada prueba con 5 puntos de partida distintos y medimos con el conjunto de **validación**.

In [ ]:
clases = sorted(y.unique())
a_numero = {c: i for i, c in enumerate(clases)}
ytr, yva, yte = y_train.map(a_numero), y_val.map(a_numero), y_test.map(a_numero)

for nombre, escalar in [("CON escalado", True), ("SIN escalado", False)]:
    A, B = listo(X_train, escalar), listo(X_val, escalar)
    aciertos = []
    for semilla in range(5):
        red = MLPClassifier(hidden_layer_sizes=(8,), max_iter=300, random_state=semilla)
        red.fit(A, ytr)
        aciertos.append(red.score(B, yva))
    print(f"{nombre}: acierta en promedio el {np.mean(aciertos)*100:.1f} % de la validación")

Los números exactos pueden variar un poco según la versión de las herramientas, pero la diferencia es enorme: **la misma red, con los mismos datos, aprende o fracasa según cómo se preparó la entrada**.

## El examen final, una sola vez
Solo cuando terminamos de decidir todo, usamos el conjunto de **prueba**. Una sola vez.

In [ ]:
red = MLPClassifier(hidden_layer_sizes=(8,), max_iter=300, random_state=0).fit(Xtr, ytr)
print(f"Acierto en la prueba: {red.score(Xte, yte)*100:.1f} %")

## Retos para ti
1. En el paso 2, en lugar de marcar el sexo como «Desconocido», elimina esas filas. ¿Cuántos pingüinos pierdes?
2. Quita la columna `island` de las características. ¿Sigue acertando la red?
3. Cambia `random_state=42` por otro número en el paso 5. ¿Cambian mucho las proporciones de cada grupo?
4. **Para pensar:** si estos pingüinos fueran fotos de hojas de papa, ¿qué sería la etiqueta y quién debería ponerla?

---
*Datos: Horst, A. M., Hill, A. P. y Gorman, K. B. (2020). palmerpenguins. Recolectados por la Dra. Kristen Gorman en la Estación Palmer, Antártida.*